### Agentic RAG

#### Q1: How many lesson pages

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Parsed {len(documents)} documents.")

Parsed 72 documents.


#### Q2: Indexing and searching

In [8]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [9]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={"content": 2.0},
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

#### Q3: RAG

In [102]:
from rag_helper import RAGBase


class myRAGBase(RAGBase):
    def __init__(self, index, llm_client, instructions, model="GLM-4.7-Flash"):
        super().__init__(index, llm_client, instructions)
        self.model = model
        self.response = None

    def search(self, query, num_results=5):
        boost_dict = {"content": 2.0}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict
        )
    
    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append('content: ' + doc['content'])
            lines.append('')

        return '\n'.join(lines).strip()
    
    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=input_messages
        )

        self.response = response
        return response.choices[0].message.content
    
    def usage(self):
        return {
            "input_tokens": self.response.usage.prompt_tokens,
            "output_tokens": self.response.usage.completion_tokens
        }


In [103]:
from dotenv import load_dotenv
load_dotenv()

import os
from openai import OpenAI

zai_client = OpenAI(
    api_key=os.getenv("ZAI_API_KEY"),
    base_url="https://api.z.ai/api/paas/v4/"
)

In [104]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the lesson database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = myRAGBase(
    index=index,
    llm_client=zai_client,
    instructions=custom_instructions,
    model="GLM-4.7-Flash"
)

In [105]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

Based on the context, the agentic loop keeps calling the model by wrapping the processing in a `while True` loop. It stops when the model returns a response **without any function calls**. The loop checks a `has_function_calls` flag; if this flag is `False`, the loop breaks, signaling that the model has finished asking for tools and is ready to provide a final answer.


In [106]:
assistant.usage()

{'input_tokens': 7101, 'output_tokens': 709}

#### Q4: Chunking

In [107]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [108]:
len(chunks)

295

In [112]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

#### Q5: RAG with chunking

In [113]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the lesson database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = myRAGBase(
    index=chunk_index,
    llm_client=zai_client,
    instructions=custom_instructions,
    model="GLM-4.7-Flash"
)

In [114]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

Based on the context, the agentic loop keeps calling the model through a `while True` loop. It uses an iteration counter to track the number of round-trips. The loop stops when the model returns a final answer with no more tool calls (function calls). The model decides how many times to search, and the loop continues until the model stops asking for tools.


In [115]:
assistant.usage()

{'input_tokens': 2230, 'output_tokens': 1205}

#### Q6: Turning it into an agent

In [128]:
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback

In [116]:
def search(query: str) -> dict[str, str]:
    """
    Search the lesson database for entries matching the given query.
    """
    return index.search(
        query,
        boost_dict={"content": 2.0},
        num_results=5
    )

In [118]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [119]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

In [129]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIChatCompletionsClient(model="GLM-4.7-Flash", client=zai_client)
)

In [131]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


Feature,Plain RAG,Agentic Loop
LLM's role,Receives context; just generates an answer,Decision-maker; chooses actions
Flow,Fixed: search → prompt → LLM,Flexible: LLM decides sequence
Search behavior,"Single search pass, no recovery","Can search multiple times, retry with different terms"
Recovery,"None - if search misses, LLM gets garbage","Can fix typos, try new keywords, ask clarifying questions"
Output predictability,Same input → same output,Same input → can take different paths
Complexity,"Simple, predictable","More API calls, higher latency, higher cost"
